In [6]:
from datetime import datetime
import pandas as pd
import sys
import os.path,json
bic_etl_home = os.getenv('bic_etl_home')
sys.path.insert(0, '/home/joe/work/myLibs')

from google.oauth2 import service_account
from googleapiclient.discovery import build
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials

from sodapy import Socrata


cimDatasets = {}
allDatasets=[]
cim_url_query = 'data.colorado.gov'



def getDatasetTrackerInfo(bic_etl_home):

    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
                 "https://www.googleapis.com/auth/drive.file",
                      "https://www.googleapis.com/auth/drive"]
    
    #creds = ServiceAccountCredentials.from_json_keyfile_name('../../scripts/client_secret.json',
    #    scope)
    creds = ServiceAccountCredentials.from_json_keyfile_name(os.path.join(bic_etl_home, 'general', 'scripts','client_secret.json'),scope)
    
    client = gspread.authorize(creds)
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
    dfTracker = pd.DataFrame(tracker.get_all_records(head=2))
    return dfTracker



## Check Record Counts on CIM
countCheck={}
title="Professional Lobbyist Bills in Colorado"
w4x4= "sche-yqzf"
url=f'https://data.colorado.gov/resource/{w4x4}.json?$select=count(*)'
response = requests.get(url)

#    print(f"Processing {nint} of {nrows} : {title}" )
# Check if request was successful
countCheck[title]=-1
if response.status_code == 200:
    a=json.loads(response.content)
    if len(a) > 0 and 'count' in a[0]:
        countCheck[title]=a[0]['count']
    elif len(a) > 0 and 'count_1' in a[0]:
        countCheck[title]=a[0]['count_1']  
    else:
        print("ROh ROh Scooby... did not get an a",a,title) 
else: 
    countCheck[title]=None      
    print(w4x4,title)

print(countCheck)


{'Professional Lobbyist Bills in Colorado': '1924266'}
